# Giai đoạn 3 — Huấn luyện trên GPU Google Colab

Chọn Colab GPU. Train MLP và CNN trên 54k ảnh, chọn trọng số bằng 6k ảnh validation.


In [ ]:
import sys, subprocess, json, shutil
from pathlib import Path
from uuid import uuid4
import google.colab
import torch

assert torch.cuda.is_available(), "Cần bật GPU; không chuyển sang CPU."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
PROJECT = Path("/content") / ("fashion_stage3_" + uuid4().hex[:8])
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/tung2106/Fashion_MNIST_CNN-MLP.git",
        str(PROJECT),
    ],
    check=True,
)
STAGE = PROJECT / "03_thuc_nghiem"
if not (STAGE / "main.py").is_file():  # Hỗ trợ bản cũ trên GitHub.
    STAGE = PROJECT / "03_Thuc_Nghiem_Cai_Tien"
COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
print("Commit:", COMMIT)

## 2. Kiểm tra thư viện

Kiểm tra CUDA và thư viện của phiên Colab.


In [ ]:
import torchvision, numpy, pandas, sklearn, matplotlib

print("Thư viện sẵn sàng. CUDA:", torch.cuda.is_available())

## 3. Train đầy đủ MLP rồi CNN

Train 5 epoch mỗi mô hình với Adam 0.001, batch 128. Lưu checkpoint có validation loss thấp nhất.


In [ ]:
before = (
    set((STAGE / "artifacts").glob("*")) if (STAGE / "artifacts").exists() else set()
)
with subprocess.Popen(
    [sys.executable, "-u", str(STAGE / "main.py")],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Huấn luyện lỗi; xem log phía trên.")
created = set((STAGE / "artifacts").glob("*")) - before
assert len(created) == 1
RUN = created.pop()
status = json.loads((RUN / "STATUS.json").read_text())
assert status["stage"] == "completed" and not status["test_evaluated"]
(RUN / "source_commit.txt").write_text(COMMIT)
print("TRAIN_COMPLETE:", RUN)

## 4. Đọc kết quả validation

So sánh loss và accuracy của từng epoch trên validation.


In [ ]:
from IPython.display import display, Image

for name in ["MLP", "CNN"]:
    print(name, json.loads((RUN / "metrics" / f"{name}_training.json").read_text()))
    display(pandas.read_csv(RUN / "metrics" / f"{name}_history.csv"))
    display(Image(filename=str(RUN / "figures" / f"{name}_learning_curve.png")))

## 5. Lưu kết quả về máy

Tải ZIP để giữ trọng số, cấu hình và lịch sử huấn luyện.


In [ ]:
from google.colab import files

archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
print("Kết quả:", archive)
files.download(archive)